# سبق 01 - اے آئی ایجنٹس کا تعارف

**ابتدائیوں کے لیے اے آئی ایجنٹس** کورس کے پہلے سبق میں خوش آمدید!

ایک **اے آئی ایجنٹ** ایک پروگرام ہے جو بڑے زبان کے ماڈل (LLM) کو اپنے استدلال کے انجن کے طور پر استعمال کرتا ہے اور حقیقی دنیا میں *عمل* کر سکتا ہے — APIs کو کال کرنا، ڈیٹا بیسز سے سوالات کرنا، یا کوڈ چلانا — تاکہ صارف کی طرف سے کسی مقصد کو پورا کیا جا سکے۔

اس نوٹ بک میں آپ اپنا پہلا ایجنٹ بنائیں گے: ایک **ٹریول ایجنٹ** جو تعطیلات کی منزلوں کی سفارش کرتا ہے۔ اس دوران آپ سیکھیں گے کہ:

1. **Microsoft Agent Framework** کا استعمال کرتے ہوئے Azure AI Foundry Agent Service سے کیسے منسلک ہوں۔
2. ایجنٹ کو ایک **ٹول** دیں — ایک سادہ پائتھن فنکشن جسے وہ کال کر سکتا ہے۔
3. ایجنٹ کو چلائیں اور اس کے جواب کا جائزہ لیں۔
4. ایجنٹ کے جواب کو ٹوکن بہ ٹوکن اسٹریم کریں۔


## سیٹ اپ

اس نوٹ بک کو چلانے سے پہلے، یقینی بنائیں کہ آپ کے پاس:

1. **ایک Azure AI Foundry پروجیکٹ** جس میں چیٹ ماڈل تعینات ہو (مثلاً `gpt-4o-mini`)۔
2. **Azure CLI میں لاگ ان ہو چکے ہیں** — اپنے ٹرمینل میں `az login` چلائیں۔
3. **ضروری ماحول متغیرات سیٹ کیے ہیں:**
   - `AZURE_AI_PROJECT_ENDPOINT` — آپ کے Azure AI Foundry پروجیکٹ کا اینڈ پوائنٹ۔
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — آپ کے تعینات کردہ ماڈل کا نام۔

نیچے والا سیل آپ کو درکار Python پیکجز انسٹال کرتا ہے۔


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## اپنا پہلا ایجنٹ بنانا

ایک ایجنٹ کو دو چیزوں کی ضرورت ہوتی ہے:

- **ہدایات** جو اسے بتائیں کہ *وہ کون ہے* اور *کیسا برتاؤ کرنا ہے* (ایک سسٹم پرامپٹ)۔
- **ٹولز** — پائیتھن کے فنکشنز جن پر `@tool` کی ڈیکوریٹر لگا ہو، جنہیں ایجنٹ معلومات حاصل کرنے یا کارروائیاں انجام دینے کے لیے بلا سکتا ہے۔

نیچے ہم ایک سادہ ٹول کی تعریف کرتے ہیں جو مشہور تعطیلات کے مقامات کی فہرست واپس کرتا ہے۔ جب کوئی صارف سفر کی سفارشات مانگے گا تو ایجنٹ اس ٹول کو استعمال کرے گا۔


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## اسٹریمنگ جوابات

زیادہ انٹرایکٹو تجربے کے لیے آپ ایجنٹ کے جواب کو **اسٹریمنگ** کر سکتے ہیں۔ پورے جواب کا انتظار کرنے کے بجائے، ایجنٹ جتنا متن تیار کرتا ہے وہ جزوی طور پر دیتا ہے۔ یہ خصوصاً چیٹ انٹرفیسز میں مفید ہوتا ہے جہاں آپ حقیقی وقت میں آؤٹ پٹ دکھانا چاہتے ہیں۔


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ کیسے:

- **ایک فراہم کنندہ بنائیں** جو `AzureAIProjectAgentProvider` کے ذریعے Azure AI Foundry Agent Service سے جڑتا ہے۔
- **ایک ٹول کی تعریف کریں** `@tool` ڈیکوریٹر استعمال کرتے ہوئے تاکہ ایجنٹ آپ کے پائتھون افعال کو کال کر سکے۔
- **ایجنٹ چلائیں** صارف کے پیغام کے ساتھ اور اس کا جواب پرنٹ کریں۔
- **جوابات کو اسٹریم کریں** تاکہ حقیقی وقت میں نتائج حاصل ہوں۔

اگلے سبق میں ہم ایجنٹیک فریم ورکس کو مزید گہرائی میں دیکھیں گے اور سیکھیں گے کہ ایجنٹ کو زیادہ طاقتور ٹولز اور کثیر مرحلہ منطق کی صلاحیتیں کیسے دیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ذمہ داری سے مستثنیٰ**:  
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعہ ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا ناکامیاں ہو سکتی ہیں۔ اصل دستاویز اپنی مادری زبان میں معتبر ماخذ سمجھی جانی چاہیے۔ اہم معلومات کے لیے پیشہ ورانہ انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے ہونے والی کسی بھی غلط فہمی یا غلط تعبیر کی ذمہ داری ہم نہیں لیتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
